# CV02 — Kiểm định tính dừng ADF & Sai phân bậc $d$

**Mã công việc:** CV02  
**Phụ trách:** Phương — Nhóm 7  
**Học phần:** Phân tích Dữ liệu Chuỗi Thời gian và Dự báo — UTH  
**Dữ liệu:** `data/processed/train.csv` (163 tháng: 07/1991 – 01/2005)

---
## Mục tiêu
1. Kiểm định ADF trên chuỗi Train gốc → xác nhận **không dừng** ($p \geq 0.05$).
2. Lấy sai phân bậc 1 ($d=1$): $\Delta y_t = y_t - y_{t-1}$.
3. Kiểm định ADF lần 2 → xác nhận **có thể dừng** sau sai phân.
4. Vẽ biểu đồ so sánh chuỗi trước và sau sai phân.
5. Trả lời câu hỏi lý luận Bài 2 (1,2,3) và Bài 3 (1,2).

---
## Ràng buộc bắt buộc
- **[1]** CHỈ sử dụng `data/processed/train.csv` — KHÔNG đọc `test.csv`.
- **[2]** Kiểm định ADF chỉ trên tập TRAIN — không rò rỉ thông tin Test.
- **[3]** Mức ý nghĩa chuẩn: $\alpha = 0.05$.
- **[4]** Lưu hình ảnh với `dpi=150` và `bbox_inches='tight'`.
- **[5]** Xuất bảng ADF đầy đủ: ADF Statistic, $p$-value, Critical Values ($1\%$, $5\%$, $10\%$).

---

## 0. Import thư viện & Thiết lập đường dẫn

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller

ROOT      = os.path.abspath(os.path.join(os.getcwd(), ".."))
PROC_DIR  = os.path.join(ROOT, "data", "processed")
FIG_DIR   = os.path.join(ROOT, "outputs", "figures")
TABLE_DIR = os.path.join(ROOT, "outputs", "tables")
TRAIN_CSV = os.path.join(PROC_DIR, "train.csv")
OUT_DIFF  = os.path.join(PROC_DIR, "train_differenced.csv")

os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TABLE_DIR, exist_ok=True)
print(f"ROOT: {ROOT}")
print(f"Train CSV: {TRAIN_CSV}")

## 1. Nạp chuỗi Train

In [ ]:
df = pd.read_csv(TRAIN_CSV, index_col="date", parse_dates=True)
train = pd.to_numeric(df["value"], errors="raise")
train.index = pd.DatetimeIndex(train.index)
train = train.sort_index()
train.name = "value"

print(f"Số tháng Train: {len(train)}")
print(f"Từ: {train.index[0].strftime('%Y-%m')} đến {train.index[-1].strftime('%Y-%m')}")
print(f"\nThống kê mô tả:")
print(train.describe().round(4))

## 2. Kiểm định ADF lần 1 — Chuỗi Train gốc

> **Giả thuyết:**  
> - $H_0$: Chuỗi có nghiệm đơn vị (Unit Root) $\Rightarrow$ Chuỗi **KHÔNG dừng**  
> - $H_1$: Chuỗi không có nghiệm đơn vị $\Rightarrow$ Chuỗi **DỪng**  
>
> **Quy tắc:** Nếu $p$-value $< 0.05$ thì bác bỏ $H_0$ (chuỗi dừng). Ngược lại ($p \geq 0.05$) không đủ cơ sở bác bỏ $H_0$.

In [ ]:
result_raw = adfuller(train.dropna(), autolag="AIC")
crit = result_raw[4]

print("=" * 60)
print("KIỂM ĐỊNH ADF — CHUỖI TRAIN GỐC")
print("=" * 60)
print(f"ADF Statistic  : {result_raw[0]:.4f}")
print(f"p-value        : {result_raw[1]:.4f}")
print(f"Số lags (AIC)  : {result_raw[2]}")
print("Critical Values:")
for k, v in crit.items():
    print(f"   {k}: {v:.4f}")

is_stationary_raw = result_raw[1] < 0.05
print(f"\n=> Kết luận: Chuỗi {'DỪng ✓' if is_stationary_raw else 'KHÔNG DỪng ✗'} (p={result_raw[1]:.4f})")

res_raw_dict = {
    "series": "Chuỗi gốc",
    "adf_stat": float(result_raw[0]),
    "p_value": float(result_raw[1]),
    "n_lags": int(result_raw[2]),
    "crit_1pct": float(crit["1%"]),
    "crit_5pct": float(crit["5%"]),
    "crit_10pct": float(crit["10%"]),
    "is_stationary": is_stationary_raw
}

### Trả lời Câu hỏi Bài 2

**Câu 1 — Giả thuyết vô hiệu ($H_0$) và giả thuyết đối ($H_1$) của kiểm định ADF là gì?**  
- $H_0$: Chuỗi tồn tại nghiệm đơn vị (Unit Root) $\Rightarrow$ Chuỗi **không dừng** (Non-Stationary). Trung bình và/hoặc phương sai thay đổi theo thời gian.
- $H_1$: Chuỗi không có nghiệm đơn vị $\Rightarrow$ Chuỗi **dừng** (Stationary). Các thống kê cơ bản ổn định theo thời gian.

**Câu 2 — Giá trị $p$-value phản ánh điều gì? Với mức ý nghĩa $\alpha = 0.05$, quy tắc bác bỏ $H_0$ như thế nào?**  
- $p$-value là xác suất quan sát được một mẫu dữ liệu có giá trị thống kê ADF cực đoan ít nhất như vậy, **giả định $H_0$ là đúng**.
- Nếu $p < 0.05$: Bác bỏ $H_0$ $\Rightarrow$ Chuỗi dừng.
- Nếu $p \geq 0.05$: Không đủ bằng chứng thống kê để bác bỏ $H_0$ $\Rightarrow$ Chuỗi không dừng.

**Câu 3 — Chuỗi dữ liệu gốc có thỏa mãn tính dừng không?**  
- **Không.** ADF Statistic $= +2.2309 > -2.8813$ (ngưỡng $5\%$) và $p$-value $= 0.9989 \gg 0.05$. Không đủ cơ sở bác bỏ $H_0$. Chuỗi a10 gốc mang **xu hướng tăng dài hạn** (trend) và **biên độ mùa vụ mở rộng dần** (phương sai tăng) — đây là chuỗi **không dừng**.

## 3. Sai phân bậc 1 ($d=1$): $\Delta y_t = y_t - y_{t-1}$

In [ ]:
train_diff = train.diff(1).dropna()
train_diff.name = "value"

# Lưu file deliverable
train_diff.to_csv(OUT_DIFF, index_label="date")

print(f"Chuỗi sau sai phân bậc 1: {len(train_diff)} quan sát")
print(f"Từ: {train_diff.index[0].strftime('%Y-%m')} đến {train_diff.index[-1].strftime('%Y-%m')}")
print(f"\nMean  : {train_diff.mean():.4f}")
print(f"Std   : {train_diff.std():.4f}")
print(f"Min   : {train_diff.min():.4f}")
print(f"Max   : {train_diff.max():.4f}")
print(f"\n[OK] Đã lưu: {OUT_DIFF}")

## 4. Kiểm định ADF lần 2 — Chuỗi sau sai phân

In [ ]:
result_diff = adfuller(train_diff.dropna(), autolag="AIC")
crit2 = result_diff[4]

print("=" * 60)
print("KIỂM ĐỊNH ADF — CHUỖI SAI PHÂN BẬC 1")
print("=" * 60)
print(f"ADF Statistic  : {result_diff[0]:.4f}")
print(f"p-value        : {result_diff[1]:.4f}")
print(f"Số lags (AIC)  : {result_diff[2]}")
print("Critical Values:")
for k, v in crit2.items():
    print(f"   {k}: {v:.4f}")

is_stationary_diff = result_diff[1] < 0.05
print(f"\n=> Kết luận: Chuỗi sai phân {'DỪng ✓' if is_stationary_diff else 'KHÔNG DỪng ✗'} (p={result_diff[1]:.4f})")

res_diff_dict = {
    "series": "Chuỗi sai phân d=1",
    "adf_stat": float(result_diff[0]),
    "p_value": float(result_diff[1]),
    "n_lags": int(result_diff[2]),
    "crit_1pct": float(crit2["1%"]),
    "crit_5pct": float(crit2["5%"]),
    "crit_10pct": float(crit2["10%"]),
    "is_stationary": is_stationary_diff
}

### Trả lời Câu hỏi Bài 3

**Câu 1 — Sau khi lấy sai phân bậc 1, chuỗi đã đạt tính dừng chưa ($p$-value $< 0.05$)?**  
- **Phân tích:** Với `autolag='AIC'`, bộ chọn lag tự động chọn **13 lags** (bao quát hơn 1 chu kỳ mùa vụ 12 tháng) để kiểm tra nghiệm đơn vị mùa vụ. Kết quả: ADF $= -1.7061$, $p = 0.4280 > 0.05$.
- Điều này cho thấy sai phân thường bậc 1 đã **triệt tiêu xu hướng tuyến tính** nhưng **còn tồn tại thành phần nghiệm đơn vị mùa vụ** (seasonal unit root) tại chu kỳ $s = 12$.
- Đây là lý do mô hình **SARIMA** (CV05) bổ sung thêm sai phân mùa vụ $D=1$ để triệt tiêu hoàn toàn tự tương quan tại lag $12, 24$.

**Câu 2 — Nếu chuỗi đã dừng (sai phân thường), giá trị tham số $d$ trong ARIMA là bao nhiêu?**  
- **$d = 1$**: Chỉ cần lấy sai phân bậc 1 một lần để triệt tiêu xu hướng tuyến tính dài hạn của chuỗi doanh số bán thuốc a10. Tham số $d$ trong $\text{ARIMA}(p, 1, q)$ phản ánh đúng bậc sai phân này.

## 5. Vẽ biểu đồ so sánh Trước & Sau sai phân

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=False)

# --- Subplot 1: Chuỗi gốc ---
ax1.plot(train.index, train, color="steelblue", linewidth=1.2)
ax1.set_title(
    f"Chuỗi Train gốc — KHÔNG DỪng (ADF={result_raw[0]:.4f}, p={result_raw[1]:.4f})",
    fontsize=11
)
ax1.set_ylabel("Doanh số (triệu AUD)")
ax1.grid(alpha=0.3)

# --- Subplot 2: Chuỗi sau sai phân ---
ax2.plot(train_diff.index, train_diff, color="darkorange", linewidth=1.2)
ax2.axhline(0, linestyle="--", color="gray", alpha=0.7, label="y=0")
ax2.set_title(
    f"Chuỗi sau Sai phân bậc 1 (d=1) — ADF={result_diff[0]:.4f}, p={result_diff[1]:.4f}",
    fontsize=11
)
ax2.set_ylabel("$\Delta$Doanh số")
ax2.set_xlabel("Thời gian")
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3)

plt.suptitle("CV02 — So sánh Chuỗi Gốc vs Chuỗi Sai phân bậc 1", fontsize=13, fontweight="bold")
plt.tight_layout()

out_fig = os.path.join(FIG_DIR, "cv02_adf_before_after_differencing.png")
fig.savefig(out_fig, dpi=150, bbox_inches="tight")
plt.show()
print(f"[OK] Đã lưu: {out_fig}")

## 6. Lưu bảng kết quả ADF

In [ ]:
summary_df = pd.DataFrame([res_raw_dict, res_diff_dict])
out_tbl = os.path.join(TABLE_DIR, "cv02_adf_test_summary.csv")
summary_df.to_csv(out_tbl, index=False)

print("=" * 60)
print("BẢNG TÓM TẮT KẾT QUẢ KIỂM ĐỊNH ADF")
print("=" * 60)
print(summary_df.to_string(index=False))
print(f"\n[OK] Đã lưu: {out_tbl}")
print("\n[DONE] CV02 hoàn tất. Chuyển train_differenced.csv cho CV03.")